# GPU Figures: 4.3 Coarsening, 4.8 Spinodal, 4.6 Convergence

This notebook embeds the GPU-first scripts (with CPU fallback).
Run this notebook from the repository root. Generated figures are written to `output/`. For long simulations, start with the CPU coarsening example before running the full spinodal or convergence studies.


## Figure 4.3 Coarsening (GPU/CPU fallback)


In [ ]:
"""Coarsening test for Figure 4.3 (Cahn-Hilliard-Darcy) – GPU-first (CuPy), CPU fallback."""

import os
import numpy as np
import matplotlib.pyplot as plt

# Backend selection
cp = None
USE_GPU = False
try:
    import cupy as cp
    from cupyx.scipy.fft import fft2 as fft2_wrap, ifft2 as ifft2_wrap, fftfreq as fftfreq_wrap
    xp = cp
    USE_GPU = True
    print("Using CuPy GPU backend")
except ImportError:
    from scipy.fft import fft2 as fft2_wrap, ifft2 as ifft2_wrap, fftfreq as fftfreq_wrap
    xp = np
    print("Using CPU backend")


def to_cpu(arr):
    if USE_GPU and cp is not None:
        return cp.asnumpy(arr)
    return arr


# Ensure output directory exists
os.makedirs("output", exist_ok=True)


class CahnHilliardDarcySolver:
    """Corrected Solver for the Cahn-Hilliard-Darcy system (Yang 2021)."""

    def __init__(self, Lx=2*np.pi, Ly=2*np.pi, Nx=128, Ny=128, dt=0.001,
                 alpha=100.0, M=1.0, lambda_param=0.01,
                 epsilon=0.05, S=2.0, tau=1.0, B=10.0):

        # Parameters
        self.Lx, self.Ly = Lx, Ly
        self.Nx, self.Ny = Nx, Ny
        self.dt = dt
        self.alpha = alpha
        self.M = M
        self.lam = lambda_param
        self.eps = epsilon
        self.S = S
        self.tau = tau
        self.B = B

        # Spatial grid and wavenumbers
        self.dx = Lx / Nx
        self.dy = Ly / Ny
        self.x = xp.linspace(0, Lx, Nx, endpoint=False)
        self.y = xp.linspace(0, Ly, Ny, endpoint=False)
        self.X, self.Y = xp.meshgrid(self.x, self.y, indexing='ij')

        # Spectral Grid
        self.kx = 2 * xp.pi * fftfreq_wrap(Nx, d=self.dx)
        self.ky = 2 * xp.pi * fftfreq_wrap(Ny, d=self.dy)
        self.KX, self.KY = xp.meshgrid(self.kx, self.ky, indexing='ij')
        self.K2 = self.KX**2 + self.KY**2
        self.K2[0,0] = 1e-10  # Avoid division by zero

        # Fields
        self.phi = xp.zeros((Nx, Ny))
        self.phi_old = xp.zeros((Nx, Ny))
        self.u = xp.zeros((Nx, Ny))
        self.v = xp.zeros((Nx, Ny))
        self.p = xp.zeros((Nx, Ny))
        self.mu = xp.zeros((Nx, Ny))

        # SAV variable U
        self.U = xp.array(0.0)
        self.t = 0.0

    def init_two_circles(self):
        """Initial condition: two tanh-profile disks (Fig. 4.3)."""
        x1, y1 = np.pi - 0.8, np.pi
        x2, y2 = np.pi + 1.7, np.pi
        r1, r2 = 1.4, 0.5

        dist1 = xp.sqrt((self.X - x1)**2 + (self.Y - y1)**2)
        dist2 = xp.sqrt((self.X - x2)**2 + (self.Y - y2)**2)

        # Tanh profile setup
        self.phi = 1.0 + xp.tanh((r1 - dist1)/(1.5*self.eps)) + \
                         xp.tanh((r2 - dist2)/(1.5*self.eps))
        self.phi_old = self.phi.copy()
        self._init_sav()

    def _init_sav(self):
        F = (0.25 / self.eps**2) * (self.phi**2 - 1)**2
        E_bulk = xp.sum(F) * self.dx * self.dy
        self.U = xp.sqrt(E_bulk + self.B)

    def step(self):
        """Perform one SAV time step (Cahn-Hilliard + Darcy)."""
        # --- Cahn-Hilliard step ---
        phi_star = 2.0 * self.phi - self.phi_old
        f_phi = (1.0 / self.eps**2) * (phi_star**3 - phi_star)
        F_term = (0.25 / self.eps**2) * (phi_star**2 - 1)**2
        E_integral = xp.sum(F_term) * self.dx * self.dy
        H = self.lam * f_phi / xp.sqrt(E_integral + self.B)

        phi_hat = fft2_wrap(self.phi)
        phi_old_hat = fft2_wrap(self.phi_old)
        grad_phi_x = xp.real(ifft2_wrap(1j * self.KX * phi_hat))
        grad_phi_y = xp.real(ifft2_wrap(1j * self.KY * phi_hat))
        advection = self.u * grad_phi_x + self.v * grad_phi_y
        adv_hat = fft2_wrap(advection)
        stab_coeff = self.S / self.eps**2
        lhs_op = (1.5/self.dt) + self.M * self.lam * self.K2**2 + self.M * stab_coeff * self.K2
        rhs_time = (2.0 * phi_hat - 0.5 * phi_old_hat) / self.dt
        forcing_spatial = H * self.U - stab_coeff * phi_star
        forcing_hat = fft2_wrap(forcing_spatial)
        rhs_spatial = -self.M * self.K2 * forcing_hat
        phi_new_hat = (rhs_time - adv_hat + rhs_spatial) / lhs_op
        phi_new = xp.real(ifft2_wrap(phi_new_hat))
        diff_phi = phi_new - self.phi
        self.U += 0.5 * xp.sum(H * diff_phi) * self.dx * self.dy

        # --- Darcy step ---
        lap_phi_new = xp.real(ifft2_wrap(-self.K2 * fft2_wrap(phi_new)))
        f_phi_new = (1.0 / self.eps**2) * (phi_new**3 - phi_new)
        F_new = (0.25 / self.eps**2) * (phi_new**2 - 1)**2
        E_new = xp.sum(F_new) * self.dx * self.dy
        H_new = self.lam * f_phi_new / xp.sqrt(E_new + self.B)
        self.mu = self.lam * (-lap_phi_new + H_new * self.U)
        mu_hat = fft2_wrap(self.mu)
        grad_mu_x = xp.real(ifft2_wrap(1j * self.KX * mu_hat))
        grad_mu_y = xp.real(ifft2_wrap(1j * self.KY * mu_hat))
        force_x = -phi_new * grad_mu_x
        force_y = -phi_new * grad_mu_y
        coeff_u = (self.tau / self.dt) + self.alpha
        rhs_u = (self.tau / self.dt) * self.u + force_x
        rhs_v = (self.tau / self.dt) * self.v + force_y
        div_rhs = 1j * self.KX * fft2_wrap(rhs_u) + 1j * self.KY * fft2_wrap(rhs_v)
        p_hat = div_rhs / (-self.K2)
        p_hat[0,0] = 0.0
        grad_p_x = xp.real(ifft2_wrap(1j * self.KX * p_hat))
        grad_p_y = xp.real(ifft2_wrap(1j * self.KY * p_hat))
        self.u = (rhs_u - grad_p_x) / coeff_u
        self.v = (rhs_v - grad_p_y) / coeff_u
        self.phi_old = self.phi.copy()
        self.phi = phi_new
        self.t += self.dt

def run_case(label, solver, target_times, save_name):
    print(f"--- Running {label} ---")
    snapshots = []
    if 0.0 in target_times:
        snapshots.append((0.0, to_cpu(solver.phi.copy())))
    max_time = max(target_times)
    steps = int(max_time / solver.dt) + 20
    for _ in range(steps):
        solver.step()
        for target in target_times:
            if abs(solver.t - target) < solver.dt * 0.6:
                snapshots.append((target, to_cpu(solver.phi.copy())))
    if not snapshots:
        return
    snapshots.sort(key=lambda x: x[0])
    cols = min(len(snapshots), 6)
    rows = (len(snapshots) - 1) // 6 + 1
    fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows + 0.5))
    if rows == 1 and cols == 1:
        axes = [axes]
    elif rows > 1 or cols > 1:
        axes = axes.flatten()
    Xp = to_cpu(solver.X)
    Yp = to_cpu(solver.Y)
    for i, (t_val, phi_val) in enumerate(snapshots):
        ax = axes[i]
        ax.contourf(Xp, Yp, phi_val, levels=np.linspace(-1.1, 1.1, 50), cmap='jet')
        ax.contour(Xp, Yp, phi_val, levels=[0], colors='white', linewidths=1)
        ax.set_title(f"t={t_val:.2f}")
        ax.axis('off')
        ax.set_aspect('equal')
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    plt.suptitle(label)
    plt.tight_layout()
    plt.savefig(f"output/{save_name}.png", dpi=150)
    plt.close()
    print(f"Saved {save_name}.png")

if __name__ == "__main__":
    solver43 = CahnHilliardDarcySolver(
        Lx=2*np.pi, Ly=2*np.pi, Nx=128, Ny=128, dt=0.005,
        alpha=100.0, M=1.0, lambda_param=0.01,
        epsilon=0.05, S=2.0, tau=1.0
    )
    solver43.init_two_circles()
    times43 = [0.0, 1.4, 1.9, 1.95, 2.45, 5.0]
    run_case("Figure 4.3: Coarsening", solver43, times43, "Figure_4_3_Fixed_Color")


## Figure 4.8 Spinodal (GPU/CPU fallback)


In [ ]:
"""
GPU-first (CuPy) version of the spinodal decomposition driver for Fig. 4.8.
Falls back to CPU if CuPy/CUDA is unavailable.
"""

import os
import numpy as np
import matplotlib.pyplot as plt

# Backend selection
cp = None
USE_GPU = False
try:
    import cupy as cp
    from cupyx.scipy.fft import fft2 as fft2_gpu, ifft2 as ifft2_gpu, fftfreq as fftfreq_gpu
    USE_GPU = True
    xp_backend = cp
    fft2_wrap = fft2_gpu
    ifft2_wrap = ifft2_gpu
    fftfreq_wrap = fftfreq_gpu
    print("Using CuPy GPU backend")
except ImportError:
    from scipy.fft import fft2 as fft2_cpu, ifft2 as ifft2_cpu, fftfreq as fftfreq_cpu
    xp_backend = np
    fft2_wrap = fft2_cpu
    ifft2_wrap = ifft2_cpu
    fftfreq_wrap = fftfreq_cpu
    print("Using CPU backend")


def to_cpu(arr):
    if USE_GPU and cp is not None:
        return cp.asnumpy(arr)
    return arr


# Ensure output directory exists
os.makedirs("output", exist_ok=True)


class CahnHilliardDarcySolver:
    """
    Solver for the Cahn-Hilliard-Darcy system.
    """

    def __init__(self, Lx=2*np.pi, Ly=2*np.pi, Nx=512, Ny=512, dt=0.001,
                 alpha=100.0, M=1.0, lambda_param=0.01,
                 epsilon=0.025, S=10.0, tau=1.0, B=10.0):

        # Parameters
        self.Lx, self.Ly = Lx, Ly
        self.Nx, self.Ny = Nx, Ny
        self.dt = dt
        self.alpha = alpha
        self.M = M
        self.lam = lambda_param
        self.eps = epsilon
        self.S = S
        self.tau = tau
        self.B = B

        # Spatial Grid
        self.dx = Lx / Nx
        self.dy = Ly / Ny
        self.x = xp_backend.linspace(0, Lx, Nx, endpoint=False)
        self.y = xp_backend.linspace(0, Ly, Ny, endpoint=False)
        self.X, self.Y = xp_backend.meshgrid(self.x, self.y, indexing='ij')

        # Spectral Grid
        self.kx = 2 * xp_backend.pi * fftfreq_wrap(Nx, d=self.dx)
        self.ky = 2 * xp_backend.pi * fftfreq_wrap(Ny, d=self.dy)
        self.KX, self.KY = xp_backend.meshgrid(self.kx, self.ky, indexing='ij')
        self.K2 = self.KX**2 + self.KY**2
        self.K2[0,0] = 1e-10 # Avoid division by zero

        # Fields
        self.phi = xp_backend.zeros((Nx, Ny))
        self.phi_old = xp_backend.zeros((Nx, Ny))
        self.u = xp_backend.zeros((Nx, Ny))
        self.v = xp_backend.zeros((Nx, Ny))
        self.p = xp_backend.zeros((Nx, Ny))
        self.mu = xp_backend.zeros((Nx, Ny))

        # SAV variable U
        self.U = xp_backend.array(0.0)
        self.t = 0.0

    def init_two_circles(self):
        """Initialize for Figure 4.3"""
        x1, y1 = np.pi - 0.8, np.pi
        x2, y2 = np.pi + 1.7, np.pi
        r1, r2 = 1.4, 0.5

        dist1 = xp_backend.sqrt((self.X - x1)**2 + (self.Y - y1)**2)
        dist2 = xp_backend.sqrt((self.X - x2)**2 + (self.Y - y2)**2)

        # Tanh profile setup
        self.phi = 1.0 + xp_backend.tanh((r1 - dist1)/(1.5*self.eps)) + \
                         xp_backend.tanh((r2 - dist2)/(1.5*self.eps))
        self.phi_old = self.phi.copy()
        self._init_sav()

    def init_spinodal(self, phi_avg, noise_amp=0.001):
        """Initialize for Figure 4.8 with small random noise (paper: 0.001*(2*rand-1.0)))."""
        noise = noise_amp * (2 * xp_backend.random.rand(self.Nx, self.Ny) - 1)
        self.phi = phi_avg + noise
        self.phi_old = self.phi.copy()
        self._init_sav()

    def _init_sav(self):
        # Energy functional F(phi)
        F = (0.25 / self.eps**2) * (self.phi**2 - 1)**2
        E_bulk = xp_backend.sum(F) * self.dx * self.dy
        self.U = xp_backend.sqrt(E_bulk + self.B)

    def step(self):
        """Perform one SAV time step (Cahn-Hilliard + Darcy)."""
        # --- Cahn-Hilliard step ---

        # Extrapolate phi for nonlinearity (2nd order BDF-like)
        phi_star = 2.0 * self.phi - self.phi_old

        # A. Compute H (Nonlinear part ONLY)
        # f(phi) = (phi^3 - phi)/eps^2
        f_phi = (1.0 / self.eps**2) * (phi_star**3 - phi_star)

        # Calculate energy integral for H denominator
        F_term = (0.25 / self.eps**2) * (phi_star**2 - 1)**2
        E_integral = xp_backend.sum(F_term) * self.dx * self.dy

        # H = f(phi) / sqrt(...)
        H = f_phi / xp_backend.sqrt(E_integral + self.B)

        # B. Solve for phi^(n+1)

        phi_hat = fft2_wrap(self.phi)
        phi_old_hat = fft2_wrap(self.phi_old)

        # Advection: u � grad(phi)
        grad_phi_x = xp_backend.real(ifft2_wrap(1j * self.KX * phi_hat))
        grad_phi_y = xp_backend.real(ifft2_wrap(1j * self.KY * phi_hat))
        advection = self.u * grad_phi_x + self.v * grad_phi_y
        adv_hat = fft2_wrap(advection)

        # Stabilization coeff S
        stab_coeff = self.S / self.eps**2

        # LHS Operator (Implicit)
        lhs_op = (1.5/self.dt) + \
                 self.M * self.lam * self.K2**2 + \
                 self.M * self.lam * stab_coeff * self.K2

        # RHS:
        # 1. Time terms (BDF2: (2phi - 0.5phi_old)/dt)
        rhs_time = (2.0 * phi_hat - 0.5 * phi_old_hat) / self.dt

        # 2. Forcing from SAV
        forcing_spatial = self.lam * (H * self.U - stab_coeff * phi_star)
        forcing_hat = fft2_wrap(forcing_spatial)
        rhs_spatial = -self.M * self.K2 * forcing_hat

        rhs_total = rhs_time - adv_hat + rhs_spatial

        phi_new_hat = rhs_total / lhs_op
        phi_new = xp_backend.real(ifft2_wrap(phi_new_hat))

        # C. Update U (SAV)
        diff_phi = phi_new - self.phi
        integral_update = 0.5 * xp_backend.sum(H * diff_phi) * self.dx * self.dy
        self.U = self.U + integral_update

        # --- 2. Darcy Step ---

        # Update Chemical Potential mu for Darcy force
        lap_phi_new = xp_backend.real(ifft2_wrap(-self.K2 * fft2_wrap(phi_new)))

        # Recalculate H with new phi for best accuracy
        f_phi_new = (1.0 / self.eps**2) * (phi_new**3 - phi_new)
        F_new = (0.25 / self.eps**2) * (phi_new**2 - 1)**2
        E_new = xp_backend.sum(F_new) * self.dx * self.dy
        H_new = f_phi_new / xp_backend.sqrt(E_new + self.B)

        # mu definition requires lambda scaling on ALL terms
        self.mu = self.lam * (-lap_phi_new + H_new * self.U)

        # Solve Momentum: (tau/dt + alpha) u + grad p = RHS
        mu_hat = fft2_wrap(self.mu)
        grad_mu_x = xp_backend.real(ifft2_wrap(1j * self.KX * mu_hat))
        grad_mu_y = xp_backend.real(ifft2_wrap(1j * self.KY * mu_hat))

        force_x = -phi_new * grad_mu_x
        force_y = -phi_new * grad_mu_y

        coeff_u = (self.tau / self.dt) + self.alpha
        rhs_u = (self.tau / self.dt) * self.u + force_x
        rhs_v = (self.tau / self.dt) * self.v + force_y

        # Projection
        div_rhs = 1j * self.KX * fft2_wrap(rhs_u) + 1j * self.KY * fft2_wrap(rhs_v)
        p_hat = div_rhs / (-self.K2)
        p_hat[0,0] = 0.0

        grad_p_x = xp_backend.real(ifft2_wrap(1j * self.KX * p_hat))
        grad_p_y = xp_backend.real(ifft2_wrap(1j * self.KY * p_hat))

        self.u = (rhs_u - grad_p_x) / coeff_u
        self.v = (rhs_v - grad_p_y) / coeff_u

        # Update state
        self.phi_old = self.phi.copy()
        self.phi = phi_new
        self.t += self.dt

def run_case(label, solver, target_times, save_name, invert_colors=False):
    print(f"--- Running {label} ---")
    snapshots = []

    # Capture t=0
    if 0.0 in target_times:
        snapshots.append((0.0, to_cpu(solver.phi.copy())))

    max_time = max(target_times)
    steps = int(max_time / solver.dt) + 50

    for i in range(steps):
        solver.step()

        # Check times
        for target in target_times:
            if abs(solver.t - target) < solver.dt * 0.6:
                print(f"  Saving t={target:.2f}")
                snapshots.append((target, to_cpu(solver.phi.copy())))

        if solver.t > max_time + solver.dt:
            break

    # Visualization
    if not snapshots: return

    cols = min(len(snapshots), 5) # Figure 4.8 has 5 cols
    rows = (len(snapshots) - 1) // 5 + 1
    fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows + 0.5))

    if rows == 1 and cols == 1: axes = [axes]
    elif rows > 1 or cols > 1: axes = axes.flatten()

    # If fewer snapshots than axes, hide extras
    if len(axes) > len(snapshots):
        for j in range(len(snapshots), len(axes)):
            axes[j].axis('off')

    for i, (t_val, phi_val) in enumerate(snapshots):
        field = phi_val
        ax = axes[i]
        # Use 'jet' which gives blue (-1) to red (1) with yellow/green interface
        cmap_use = 'jet_r' if invert_colors else 'jet'
        cf = ax.contourf(solver.X.get() if USE_GPU else solver.X,
                         solver.Y.get() if USE_GPU else solver.Y,
                         field,
                         levels=np.linspace(-1.1, 1.1, 50), cmap=cmap_use)

        # Interface line (white)
        ax.contour(solver.X.get() if USE_GPU else solver.X,
                   solver.Y.get() if USE_GPU else solver.Y,
                   field, levels=[0], colors='white', linewidths=1)
        ax.set_title(f"t={t_val:.1f}")
        ax.axis('off')
        ax.set_aspect('equal')

    plt.suptitle(label)
    plt.tight_layout()
    plt.savefig(f"output/{save_name}.png", dpi=150)
    plt.close()
    print(f"Saved {save_name}.png")

if __name__ == "__main__":
    # --- Figure 4.8a ---
    # Parameters from Section 4.3
    # Nx=512, dt=0.001
    print("Initializing Figure 4.8a...")
    solver48a = CahnHilliardDarcySolver(
        Lx=2*np.pi, Ly=2*np.pi, Nx=512, Ny=512, dt=0.001,
        alpha=100.0, M=1.0, lambda_param=0.01,
        epsilon=0.025, S=10.0, tau=1.0
    )
    solver48a.init_spinodal(phi_avg=0.0, noise_amp=0.001)
    times48a = [0.5, 5.0, 10.0, 15.0, 40.0] # [cite: 751]
    # Invert colors to match article (swap red/blue mapping)
    run_case("Figure 4.8a: Spinodal (phi=0)", solver48a, times48a, "Figure_4_8a_Fixed", invert_colors=True)

    # --- Figure 4.8b ---
    print("Initializing Figure 4.8b...")
    solver48b = CahnHilliardDarcySolver(
        Lx=2*np.pi, Ly=2*np.pi, Nx=512, Ny=512, dt=0.001,
        alpha=100.0, M=1.0, lambda_param=0.01,
        epsilon=0.025, S=10.0, tau=1.0
    )
    solver48b.init_spinodal(phi_avg=0.3, noise_amp=0.001)
    times48b = [5.0, 10.0, 15.0, 40.0, 50.0] # [cite: 752]
    run_case("Figure 4.8b: Spinodal (phi=0.3)", solver48b, times48b, "Figure_4_8b_Fixed")


## Figure 4.6 Convergence (GPU/CPU fallback)


In [ ]:

"""
Minimal driver to reproduce Fig. 4.6 (DSAV vs EX-SAV vs AV).
GPU-first (CuPy) with CPU fallback; initial condition (4.1),
parameters (4.2), L2 errors vs small-step DSAV reference.
EX-SAV/AV drop SAV normalization; AV also sets S=0.
"""

import os
import numpy as np
import matplotlib.pyplot as plt

cp = None

# Backend: prefer CuPy if available; else NumPy/scipy.fft
USE_GPU = False
try:
    import cupy as cp
    from cupyx.scipy.fft import fft2 as fft2_gpu, ifft2 as ifft2_gpu, fftfreq as fftfreq_gpu

    xp_backend = cp
    fft2_wrap = fft2_gpu
    ifft2_wrap = ifft2_gpu
    fftfreq_wrap = fftfreq_gpu
    USE_GPU = True
    print("Using CuPy GPU backend")
except ImportError:
    from scipy.fft import fft2 as fft2_cpu, ifft2 as ifft2_cpu, fftfreq as fftfreq_cpu

    xp_backend = np
    fft2_wrap = fft2_cpu
    ifft2_wrap = ifft2_cpu
    fftfreq_wrap = fftfreq_cpu
    print("Using CPU backend")


def to_cpu(arr):
    if USE_GPU and cp is not None:
        return cp.asnumpy(arr)
    return arr

from pathlib import Path

# Directories
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


class CahnHilliardDarcySolver:
    """
    Solver with SAV toggle (use_sav=True for DSAV, False for EX-SAV/AV).
    Only the pieces needed for Figure 4.6 are kept.
    """

    def __init__(
        self,
        Lx=2 * np.pi,
        Ly=2 * np.pi,
        Nx=128,
        Ny=128,
        dt=0.001,
        alpha=100.0,
        M=1.0,
        lambda_param=0.01,
        epsilon=0.05,
        S=2.0,
        tau=1.0,
        B=10.0,
        use_sav=True,
    ):
        self.Lx, self.Ly = Lx, Ly
        self.Nx, self.Ny = Nx, Ny
        self.dt = dt
        self.alpha = alpha
        self.M = M
        self.lam = lambda_param
        self.eps = epsilon
        self.S = S
        self.tau = tau
        self.B = B
        self.use_sav = use_sav

        # Spatial grid
        self.dx = Lx / Nx
        self.dy = Ly / Ny
        self.x = xp_backend.linspace(0, Lx, Nx, endpoint=False)
        self.y = xp_backend.linspace(0, Ly, Ny, endpoint=False)
        self.X, self.Y = xp_backend.meshgrid(self.x, self.y, indexing="ij")

        # Spectral grid (Fourier wavenumbers)
        self.kx = 2 * xp_backend.pi * fftfreq_wrap(Nx, d=self.dx)
        self.ky = 2 * xp_backend.pi * fftfreq_wrap(Ny, d=self.dy)
        self.KX, self.KY = xp_backend.meshgrid(self.kx, self.ky, indexing="ij")
        self.K2 = self.KX**2 + self.KY**2
        self.K2[0, 0] = 1e-10  # avoid divide by zero

        # Fields
        self.phi = xp_backend.zeros((Nx, Ny))
        self.phi_old = xp_backend.zeros((Nx, Ny))
        self.u = xp_backend.zeros((Nx, Ny))
        self.v = xp_backend.zeros((Nx, Ny))
        self.mu = xp_backend.zeros((Nx, Ny))
        self.U = xp_backend.array(0.0)  # SAV variable
        self.t = 0.0

    def init_two_circles(self):
        """Initial condition (4.1): two tanh-profile disks."""
        x1, y1 = np.pi - 0.8, np.pi
        x2, y2 = np.pi + 1.7, np.pi
        r1, r2 = 1.4, 0.5
        dist1 = xp_backend.sqrt((self.X - x1) ** 2 + (self.Y - y1) ** 2)
        dist2 = xp_backend.sqrt((self.X - x2) ** 2 + (self.Y - y2) ** 2)
        self.phi = (
            1.0
            + xp_backend.tanh((r1 - dist1) / (1.5 * self.eps))
            + xp_backend.tanh((r2 - dist2) / (1.5 * self.eps))
        )
        self.phi_old = self.phi.copy()
        self._init_sav()

    def _init_sav(self):
        if not self.use_sav:
            # EX-SAV / AV: fix Q^n = 1 (no SAV normalization)
            self.U = 1.0
            return
        F = (0.25 / self.eps**2) * (self.phi**2 - 1) ** 2
        E_bulk = xp_backend.sum(F) * self.dx * self.dy
        self.U = xp_backend.sqrt(E_bulk + self.B)

    def step(self):
        """Perform one SAV time step (Cahn-Hilliard + Darcy)."""
        # Extrapolate phi for nonlinearity (2nd order BDF-like)
        phi_star = 2.0 * self.phi - self.phi_old

        # Nonlinear part (f = (phi^3-phi)/eps^2)
        f_phi = (1.0 / self.eps**2) * (phi_star**3 - phi_star)
        if self.use_sav:
            F_term = (0.25 / self.eps**2) * (phi_star**2 - 1) ** 2
            E_integral = xp_backend.sum(F_term) * self.dx * self.dy
            H = f_phi / xp_backend.sqrt(E_integral + self.B)
        else:
            # No-Q: standard nonlinear term without SAV scaling (AV sets S=0 via caller).
            H = f_phi

        phi_hat = fft2_wrap(self.phi)
        phi_old_hat = fft2_wrap(self.phi_old)

        # Advection: u · grad(phi)
        grad_phi_x = xp_backend.real(ifft2_wrap(1j * self.KX * phi_hat))
        grad_phi_y = xp_backend.real(ifft2_wrap(1j * self.KY * phi_hat))
        advection = self.u * grad_phi_x + self.v * grad_phi_y
        adv_hat = fft2_wrap(advection)

        stab_coeff = self.S / self.eps**2

        # LHS operator (implicit)
        lhs_op = (1.5 / self.dt) + self.M * self.lam * self.K2**2 + self.M * self.lam * stab_coeff * self.K2

        # RHS: time (BDF2) + forcing
        rhs_time = (2.0 * phi_hat - 0.5 * phi_old_hat) / self.dt
        forcing_spatial = self.lam * (H * self.U - stab_coeff * phi_star)
        forcing_hat = fft2_wrap(forcing_spatial)
        rhs_spatial = -self.M * self.K2 * forcing_hat
        rhs_total = rhs_time - adv_hat + rhs_spatial

        phi_new_hat = rhs_total / lhs_op
        phi_new = xp_backend.real(ifft2_wrap(phi_new_hat))

        # Update SAV
        if self.use_sav:
            diff_phi = phi_new - self.phi
            integral_update = 0.5 * xp_backend.sum(H * diff_phi) * self.dx * self.dy
            self.U += integral_update

        # Darcy step (velocity used for advection consistency)
        lap_phi_new = xp_backend.real(ifft2_wrap(-self.K2 * fft2_wrap(phi_new)))
        f_phi_new = (1.0 / self.eps**2) * (phi_new**3 - phi_new)
        if self.use_sav:
            F_new = (0.25 / self.eps**2) * (phi_new**2 - 1) ** 2
            E_new = xp_backend.sum(F_new) * self.dx * self.dy
            H_new = f_phi_new / xp_backend.sqrt(E_new + self.B)
            mu_explicit = H_new * self.U
        else:
            mu_explicit = f_phi_new  # no SAV scaling, no Q
        self.mu = self.lam * (-lap_phi_new + mu_explicit)

        mu_hat = fft2_wrap(self.mu)
        grad_mu_x = xp_backend.real(ifft2_wrap(1j * self.KX * mu_hat))
        grad_mu_y = xp_backend.real(ifft2_wrap(1j * self.KY * mu_hat))
        force_x = -phi_new * grad_mu_x
        force_y = -phi_new * grad_mu_y

        coeff_u = (self.tau / self.dt) + self.alpha
        rhs_u = (self.tau / self.dt) * self.u + force_x
        rhs_v = (self.tau / self.dt) * self.v + force_y

        div_rhs = 1j * self.KX * fft2_wrap(rhs_u) + 1j * self.KY * fft2_wrap(rhs_v)
        p_hat = div_rhs / (-self.K2)
        p_hat[0, 0] = 0.0
        grad_p_x = xp_backend.real(ifft2_wrap(1j * self.KX * p_hat))
        grad_p_y = xp_backend.real(ifft2_wrap(1j * self.KY * p_hat))

        self.u = (rhs_u - grad_p_x) / coeff_u
        self.v = (rhs_v - grad_p_y) / coeff_u

        self.phi_old = self.phi.copy()
        self.phi = phi_new
        self.t += self.dt


def l2_error(phi_num, phi_ref, dx, dy):
    diff = phi_num - phi_ref
    return np.sqrt(np.sum(diff**2) * dx * dy)


def compute_solution(dt, final_time, base_params, use_sav=True, S_override=None, label=None):
    params = base_params.copy()
    params.update(dict(dt=dt, use_sav=use_sav))
    if S_override is not None:
        params["S"] = S_override

    solver = CahnHilliardDarcySolver(**params)
    solver.init_two_circles()
    steps = int(np.ceil(final_time / dt))
    for _ in range(steps):
        solver.step()
    if label:
        print(f"  [{label}] t={solver.t:.5e}, steps={steps}")
    phi_cpu = to_cpu(solver.phi)
    return phi_cpu, solver.dx, solver.dy


def run_time_refinement_tests():
    base_params = dict(
        Lx=2 * np.pi,
        Ly=2 * np.pi,
        Nx=128,
        Ny=128,
        alpha=100.0,
        M=1.0,
        lambda_param=0.01,
        epsilon=0.05,
        S=2.0,
        tau=1.0,
        B=10.0,
    )

    # Use a longer horizon to separate DSAV vs EX-SAV; reference step remains small.
    final_time = 0.01
    dt_reference = 1e-9
    print(f"Computing DSAV reference (dt={dt_reference})...")
    phi_ref, dx_ref, dy_ref = compute_solution(
        dt_reference, final_time, base_params, use_sav=True, S_override=2.0, label="DSAV-ref"
    )

    dt_values = np.array([1e-2, 7e-3, 5e-3, 2e-3, 1e-3, 7e-4, 5e-4, 2e-4, 1e-4, 5e-5, 2e-5, 1e-5])

    errors_dsav = []
    errors_exsav = []
    errors_av = []

    for dt in dt_values:
        print(f"Running DSAV dt={dt}...")
        phi_dsav, dx, dy = compute_solution(dt, final_time, base_params, use_sav=True, S_override=2.0)
        errors_dsav.append(l2_error(phi_dsav, phi_ref, dx, dy))

        print(f"Running EX-SAV dt={dt}...")
        phi_exsav, dx, dy = compute_solution(dt, final_time, base_params, use_sav=False, S_override=2.0)
        errors_exsav.append(l2_error(phi_exsav, phi_ref, dx, dy))

        print(f"Running AV dt={dt}...")
        phi_av, dx, dy = compute_solution(dt, final_time, base_params, use_sav=False, S_override=0.0)
        errors_av.append(l2_error(phi_av, phi_ref, dx, dy))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # (a) DSAV vs EX-SAV
    axes[0].loglog(dt_values, errors_dsav, "rD-", label=r"DSAV:$\phi$")
    axes[0].loglog(dt_values, errors_exsav, "bd--", label=r"EX-SAV:$\phi$")
    axes[0].set_xlabel("Time Step")
    axes[0].set_ylabel(r"$L^2$ Error")
    axes[0].set_xlim(1e-5, 1e-2)
    axes[0].set_ylim(1e-7, 1e0)
    axes[0].legend()
    axes[0].grid(True, which="both", ls=":")
    axes[0].set_title("(a) DSAV and EX-SAV (no Q)")

    # (b) DSAV vs AV
    axes[1].loglog(dt_values, errors_dsav, "rD-", label=r"DSAV:$\phi$")
    axes[1].loglog(dt_values, errors_av, "bo-", label=r"AV:$\phi$")
    ref_y0 = errors_dsav[0]
    ref_line = ref_y0 * (dt_values / dt_values[0]) ** 2
    axes[1].loglog(dt_values, ref_line, "b--", label="Ref:slope 2")
    axes[1].set_xlabel("Time Step")
    axes[1].set_ylabel(r"$L^2$ Error")
    axes[1].set_xlim(1e-5, 1e-2)
    axes[1].set_ylim(1e-7, 1e0)
    axes[1].legend()
    axes[1].grid(True, which="both", ls=":")
    axes[1].set_title("(b) DSAV and AV (no S and Q)")

    plt.tight_layout()
    outfile = OUTPUT_DIR / "Figure_4_6_reproduction.png"
    plt.savefig(outfile, dpi=300)
    plt.close()
    print(f"Saved {outfile}")


if __name__ == "__main__":
    run_time_refinement_tests()
